# Deep Past Baseline (Data-First)

This notebook is a **baseline starter focused on data engineering first**.

It is built around the challenge notes in the project markdown files:
- document-level training, sentence-level test
- heavy transliteration notation and cleanup needs
- metric = geometric mean of corpus BLEU and chrF++

Use this notebook to produce reusable, model-agnostic training-ready data.

## Workflow
1. Load raw CSV files if available
2. Apply deterministic transliteration normalization
3. Build reproducible train/valid splits
4. Export cleaned datasets for any architecture
5. Keep evaluation utility aligned with competition metric

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import re
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import pandas as pd

In [ ]:
ROOT = Path.cwd().resolve().parent  # notebook is expected in ./notebooks
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "deep_past_data"
PROC_DIR = DATA_DIR / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "train": RAW_DIR / "train.csv",
    "test": RAW_DIR / "test.csv",
    "sample_submission": RAW_DIR / "sample_submission.csv",
    "published_texts": RAW_DIR / "published_texts.csv",
    "publications": RAW_DIR / "publications.csv",
    "bibliography": RAW_DIR / "bibliography.csv",
    "oa_lexicon": RAW_DIR / "OA_Lexicon_eBL.csv",
    "ebl_dictionary": RAW_DIR / "eBL_Dictionary.csv",
    "sentences_helper": RAW_DIR / "Sentences_Oare_FirstWord_LinNum.csv",
}

for key, path in PATHS.items():
    print(f"{key:16s} -> {'FOUND' if path.exists() else 'MISSING'} :: {path}")

In [ ]:
def read_csv_if_exists(path: Path) -> Optional[pd.DataFrame]:
    if not path.exists():
        return None
    return pd.read_csv(path)

datasets: Dict[str, Optional[pd.DataFrame]] = {name: read_csv_if_exists(path) for name, path in PATHS.items()}

for name, df in datasets.items():
    if df is None:
        continue
    print(f"\n{name}: shape={df.shape}")
    print(df.head(2))

In [ ]:
SUBSCRIPT_MAP = str.maketrans({
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
    "ₓ": "x",
})

PUNCT_TO_REMOVE = r"[!?/]"

def normalize_transliteration(text: str) -> str:
    if pd.isna(text):
        return ""

    value = str(text)

    value = value.replace("Ḫ", "H").replace("ḫ", "h")

    value = re.sub(r"\[x\]", " __GAP__ ", value)
    value = re.sub(r"\[\s*\.\.\.\s*\]\
 __BIG_GAP__ ", value)
    value = re.sub(r"\.\.\.", " __BIG_GAP__ ", value)

    value = value.translate(SUBSCRIPT_MAP)

    value = re.sub(PUNCT_TO_REMOVE, " ", value)
    value = value.replace(":", " ").replace(".", " ")

    value = value.replace("[", "").replace("]", "")

    value = value.replace("<", "").replace(">", "")

    value = value.replace("__GAP__", "<gap>")
    value = value.replace("__BIG_GAP__", "<big_gap>")

    value = re.sub(r"\s+", " ", value).strip()
    return value


def normalize_translation(text: str) -> str:
    if pd.isna(text):
        return ""
    value = str(text)
    value = value.replace("<", "").replace(">", "")
    value = re.sub(r"\s+", " ", value).strip()
    return value

In [ ]:
train_df = datasets.get("train")
test_df = datasets.get("test")
published_df = datasets.get("published_texts")

if train_df is not None:
    train_df = train_df.copy()
    train_df["transliteration_clean"] = train_df["transliteration"].map(normalize_transliteration)
    train_df["translation_clean"] = train_df["translation"].map(normalize_translation)
    print("train cleaned shape:", train_df.shape)

if test_df is not None:
    test_df = test_df.copy()
    test_df["transliteration_clean"] = test_df["transliteration"].map(normalize_transliteration)
    print("test cleaned shape:", test_df.shape)

if published_df is not None and "transliteration" in published_df.columns:
    published_df = published_df.copy()
    published_df["transliteration_clean"] = published_df["transliteration"].map(normalize_transliteration)
    print("published_texts cleaned shape:", published_df.shape)

if train_df is not None:
    display(train_df[["oare_id", "transliteration", "transliteration_clean", "translation_clean"]].head(3))

In [ ]:
def deterministic_split_by_id(
    df: pd.DataFrame,
    id_col: str,
    valid_ratio: float = 0.1,
    salt: str = "deep_past_split_v1",
) -> pd.DataFrame:
    cutoff = int(valid_ratio * 10_000)

    def is_valid(identifier: str) -> int:
        payload = f"{salt}:{identifier}".encode("utf-8")
        bucket = int(hashlib.md5(payload).hexdigest(), 16) % 10_000
        return int(bucket < cutoff)

    out = df.copy()
    out["is_valid"] = out[id_col].astype(str).map(is_valid)
    out["split"] = out["is_valid"].map({0: "train", 1: "valid"})
    return out

if train_df is not None:
    split_col = "oare_id" if "oare_id" in train_df.columns else train_df.columns[0]
    train_split_df = deterministic_split_by_id(train_df, id_col=split_col, valid_ratio=0.1)
    print(train_split_df["split"].value_counts(dropna=False))
else:
    train_split_df = None

In [ ]:
if train_split_df is not None:
    cols = [
        c
        for c in ["oare_id", "transliteration", "translation", "transliteration_clean", "translation_clean", "split"]
        if c in train_split_df.columns
    ]
    train_ready = train_split_df[cols].copy()
    train_ready.to_csv(PROC_DIR / "train_ready.csv", index=False)
    train_ready.query("split == 'train'").to_csv(PROC_DIR / "train_ready_train.csv", index=False)
    train_ready.query("split == 'valid'").to_csv(PROC_DIR / "train_ready_valid.csv", index=False)
    print("Saved:")
    print(PROC_DIR / "train_ready.csv")
    print(PROC_DIR / "train_ready_train.csv")
    print(PROC_DIR / "train_ready_valid.csv")

if test_df is not None:
    test_cols = [c for c in ["id", "text_id", "line_start", "line_end", "transliteration", "transliteration_clean"] if c in test_df.columns]
    test_ready = test_df[test_cols].copy()
    test_ready.to_csv(PROC_DIR / "test_ready.csv", index=False)
    print(PROC_DIR / "test_ready.csv")

In [ ]:
def geometric_mean_bleu_chrf(references: Iterable[str], hypotheses: Iterable[str]) -> Dict[str, float]:
    refs = list(references)
    hyps = list(hypotheses)
    try:
        import sacrebleu
    except ImportError as exc:
        raise RuntimeError("Install sacrebleu first: pip install sacrebleu") from exc

    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score
    gmean = math.sqrt(max(bleu, 0.0) * max(chrf, 0.0))
    return {"bleu": bleu, "chrfpp": chrf, "geometric_mean": gmean}


if train_split_df is not None:
    valid = train_split_df.query("split == 'valid'")
    if not valid.empty:
        dummy_pred = valid["translation_clean"].fillna("").tolist()
        dummy_ref = valid["translation_clean"].fillna("").tolist()
        scores = geometric_mean_bleu_chrf(dummy_ref, dummy_pred)
        print("Sanity metric (oracle preds):", json.dumps(scores, indent=2))

In [ ]:
def build_submission(test_frame: pd.DataFrame, predictions: Iterable[str], out_path: Path) -> pd.DataFrame:
    submission = pd.DataFrame({
        "id": test_frame["id"].values,
        "translation": list(predictions),
    })

    if submission["id"].duplicated().any():
        raise ValueError("Duplicate ids in submission")
    if submission["translation"].isna().any():
        raise ValueError("Null translations in submission")

    submission.to_csv(out_path, index=False)
    return submission


if test_df is not None and "id" in test_df.columns:
    baseline_predictions = ["" for _ in range(len(test_df))]  # replace with model outputs
    sub = build_submission(test_df, baseline_predictions, PROC_DIR / "submission_baseline.csv")
    display(sub.head())

## Next Up
- Replace deterministic split with document-aware or chronology-aware split if needed
- Add mined pseudo-parallel pairs from `publications.csv` with confidence filters
- Add model training cell(s) that consume `data/processed/train_ready_*.csv`
- Keep preprocessing identical in train/valid/test/inference